# ZHAW Commute Survey: Chi-squared test & Spearman correlation

In this exercise you analyse how the class gets to ZHAW. The data comes from the CSV export of the survey app (`/export.csv`), or, as long as there is no real data, from `simulate_responses.ipynb`. Both files have the same format.

| Column | Type | Content |
|---|---|---|
| `campus`, `transport`, `travel_pass`, `gender`, `living` | categorical | answer text |
| `door_to_door_min`, `distance_km`, `days_per_week`, `monthly_cost_chf`, `age` | numerical | one way, per week, per month, years |
| `satisfied`, `stressful`, `productive` | ordinal (Likert) | 1 = strongly disagree ... 5 = strongly agree |

- **Chi-squared test**: are two *categorical* variables associated?
- **Spearman correlation**: are two *numerical or ordinal* variables monotonically related?

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2, chi2_contingency, spearmanr, pearsonr

# Plot style settings
plt.style.use('dark_background')

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Get current working directory
print(os.getcwd())

## Read the survey data
All responses of the class are stored centrally in a Google Sheet, which can be read as CSV directly from the survey link. Set `use_class_data = False` to practise with the simulated data from `simulate_responses.ipynb` instead.

In [ ]:
SURVEY_URL = 'https://script.google.com/macros/s/AKfycbwecUh787yY-DxLBazOx-J9LY69JvpRU51NWrJZifoGtaFgslBet5FWHfh1NXoiACrmtQ/exec'
use_class_data = True

if use_class_data:
    df = pd.read_csv(SURVEY_URL + '?format=csv')
else:
    df = pd.read_csv('zhaw_commute_responses.csv')

print(df.shape)
df.head()

## Task 3a: Describe the data

In [ ]:
# How many students use which means of transport?
print(df['transport'].value_counts(), '\n')

# Median travel time, distance and costs per means of transport
df.groupby('transport')[['door_to_door_min', 'distance_km', 'monthly_cost_chf']].median()

## Task 3b: Chi-squared test: living situation vs. means of transport
Do students who live in their own flat, in a shared apartment or elsewhere (*Other*, e.g. with their parents) commute differently?

In [ ]:
# Contingency table
tab = pd.crosstab(df['living'], df['transport'])
tab

In [ ]:
# Null- and alternative hypothesis
# H0 = There is no association between the living situation and the means of transport
# H1 = There is an association between the living situation and the means of transport

# alpha
alpha = 0.05

# 1-alpha
prob = 1 - alpha

# Chi-squared test
stat, p, dof, expected = chi2_contingency(tab)

# Critical value
critical = chi2.ppf(prob, dof)

# Print results of the Chi-square test
print('Expected frequencies:', '\n', pd.DataFrame(expected, index=tab.index, columns=tab.columns).round(1), '\n')
print('Smallest expected frequency:', expected.min().round(2))
print('Critical value:', critical.round(4))
print('Test statistic:', stat.round(4))
print('Degrees of freedom:', dof)
print(f'p-value: {p:.4f}')

### Assumption: expected frequencies of at least 5
The chi-squared test is only reliable if all expected frequencies are at least 5 (rule of thumb). If some are smaller, merge similar categories, e.g. *Bike or e-bike* and *On foot* into *Bike / on foot*, and repeat the test.

In [ ]:
df['transport_grouped'] = df['transport'].replace({'Bike or e-bike': 'Bike / on foot', 'On foot': 'Bike / on foot'})

tab = pd.crosstab(df['living'], df['transport_grouped'])
stat, p, dof, expected = chi2_contingency(tab)

print(tab, '\n')
print('Smallest expected frequency:', expected.min().round(2))
print(f'Test statistic: {stat:.4f}, dof: {dof}, p-value: {p:.4f}')

### Effect size: Cramér's V
The p-value only tells us *whether* there is an association, not *how strong* it is. Cramér's V ranges from 0 (no association) to 1 (perfect association):

$$V = \sqrt{\frac{\chi^2}{n \cdot (\min(r, c) - 1)}}$$

In [ ]:
n = tab.to_numpy().sum()
cramers_v = np.sqrt(stat / (n * (min(tab.shape) - 1)))
print(f"Cramér's V: {cramers_v:.3f}")

In [ ]:
# Conclusion
print('Reject H0' if p < alpha else 'Do not reject H0')
# The row percentages show the pattern: students in the category 'Other' (e.g. living with their
# parents) live further away and take the train more often; students in a shared apartment often walk or cycle.
(pd.crosstab(df['living'], df['transport_grouped'], normalize='index') * 100).round(1)

## Task 3c: Chi-squared test with a Likert item: satisfaction vs. means of transport
Are some means of transport associated with higher satisfaction? With five Likert categories many expected frequencies would be small, so we first **collapse** the item into *disagree (1-2) / neutral (3) / agree (4-5)*.

In [ ]:
df['satisfied_cat'] = pd.cut(df['satisfied'], bins=[0, 2, 3, 5], labels=['Disagree', 'Neutral', 'Agree'])

tab_sat = pd.crosstab(df['transport_grouped'], df['satisfied_cat'])
stat, p, dof, expected = chi2_contingency(tab_sat)

print(tab_sat, '\n')
print('Row percentages:\n', (pd.crosstab(df['transport_grouped'], df['satisfied_cat'], normalize='index') * 100).round(1), '\n')
print('Smallest expected frequency:', expected.min().round(2))
print(f'Test statistic: {stat:.4f}, dof: {dof}, p-value: {p:.4f}')
print('Reject H0' if p < alpha else 'Do not reject H0')

### Chi-squared test: gender vs. means of transport
Do women and men use different means of transport?

In [ ]:
tab_gender = pd.crosstab(df['gender'], df['transport_grouped'])
stat, p, dof, expected = chi2_contingency(tab_gender)

print(tab_gender, '\n')
print('Smallest expected frequency:', expected.min().round(2))
print(f'Test statistic: {stat:.4f}, dof: {dof}, p-value: {p:.4f}')
print('Reject H0' if p < alpha else 'Do not reject H0')
# Not rejecting H0 does not prove that there is no difference: the data only gives no
# evidence of an association at the 5% level.

## Task 3d: Spearman correlation
**Why Spearman instead of Pearson?**
- The Likert items are *ordinal*: only the order of the answers is meaningful, not the distances.
- Travel time, distance and costs are *right-skewed* with a few extreme values, which strongly influence Pearson's r.
- Spearman's rank correlation $\rho$ correlates the *ranks* of the values. It measures whether the relationship is *monotonic* (not necessarily linear) and is robust against outliers.

In [ ]:
# Scatter plot: distance vs. travel time
plt.figure(figsize=(7, 5))
plt.scatter(df['distance_km'], df['door_to_door_min'], c='#5cc2cc', s=15)
plt.xlabel('Distance one way (km)')
plt.ylabel('Door to door one way (minutes)')
plt.title('Distance vs. travel time')
plt.show()

# Spearman's rho with p-value
rho, p = spearmanr(df['distance_km'], df['door_to_door_min'])
print(f"Spearman's rho: {rho:.3f} (p-value: {p:.4f})")

In [ ]:
# H0: rho = 0 (no monotonic relationship between travel time and satisfaction)
# H1: rho != 0
rho, p = spearmanr(df['door_to_door_min'], df['satisfied'])
print(f"Travel time vs. satisfaction: Spearman's rho = {rho:.3f} (p-value: {p:.4f})")

rho, p = spearmanr(df['distance_km'], df['monthly_cost_chf'])
print(f"Distance vs. monthly costs:   Spearman's rho = {rho:.3f} (p-value: {p:.4f})")

### Correlation matrix

In [ ]:
cols = ['door_to_door_min', 'distance_km', 'days_per_week', 'monthly_cost_chf', 'age', 'satisfied', 'stressful', 'productive']
rho_matrix = df[cols].corr(method='spearman')

plt.figure(figsize=(8, 6))
sns.heatmap(rho_matrix, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, center=0, square=True,
            linewidths=0.5, cbar_kws={'label': "Spearman's rho"})
plt.title('Spearman correlations')
plt.show()

### Spearman vs. Pearson

In [ ]:
rho, _ = spearmanr(df['distance_km'], df['monthly_cost_chf'])
r, _ = pearsonr(df['distance_km'], df['monthly_cost_chf'])
print(f"Spearman's rho: {rho:.3f}")
print(f"Pearson's r:    {r:.3f}")
# The two coefficients differ because the relationship is monotonic but not linear (a travel pass has
# a fixed price, car costs grow with the distance) and a few long trips are extreme values.

# Interpretation of the results
Summarise in a few sentences:
- Is the means of transport associated with the living situation, with satisfaction and with gender? How strong is the association?
- Which variables are correlated, in which direction, and how strongly?
- Which limitations do you see (sample size, merged categories, self-reported times and costs, correlation vs. causation)?

In [ ]:
# Conclusion


### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')